# AI-based Predictive Maintenance for Prediction of Simulated Turbofan Engine

**NASA C-MAPSS FD001, LSTM-based RUL prediction**

Protocol: engine-wise 80/20 split; MinMax scaler fit on **training-split engines only**; train targets `min(RUL, 125)`.

**Mean signed error** = mean(pred − true): **positive** = late (over-predicting remaining life); **negative** = early.

**NASA score:** with d = pred − true, sum of exp(−d/13)−1 if d<0 else exp(d/10)−1 (compare only on the same evaluation target).


In [1]:
"model" in globals()

False

In [ ]:
print("Model exists:", "model" in globals())
if "model" in globals():
    print(type(model))

Model exists: False


In [4]:

from pathlib import Path
import json
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings("ignore")

cwd = Path.cwd()
PROJECT_ROOT = None
for folder in [cwd, *cwd.parents]:
    if (folder / "data" / "raw" / "train_FD001.txt").exists():
        PROJECT_ROOT = folder
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError("train_FD001.txt not found under data/raw")

REPORTS = PROJECT_ROOT / "reports"
FIG_DIR = REPORTS / "figures"
REPORTS.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)
LOG_PATH = REPORTS / "run_log.txt"
LOG_PATH.write_text("", encoding="utf-8")

def log(msg: str) -> None:
    print(msg)
    with LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(msg + "\n")

log(f"Project root: {PROJECT_ROOT}")


Project root: c:\Users\samya\Desktop\mini_project


In [5]:

DATA_DIR = PROJECT_ROOT / "data" / "raw"
RUL_CAP = 125
WINDOW = 30

columns = (
    ["unit_id", "cycle"]
    + [f"setting_{i}" for i in range(1, 4)]
    + [f"sensor_{i}" for i in range(1, 22)]
)

train_df = pd.read_csv(DATA_DIR / "train_FD001.txt", sep=r"\s+", header=None, names=columns)
test_df = pd.read_csv(DATA_DIR / "test_FD001.txt", sep=r"\s+", header=None, names=columns)
rul_df = pd.read_csv(DATA_DIR / "RUL_FD001.txt", sep=r"\s+", header=None, names=["RUL"])

sensor_cols = [c for c in train_df.columns if c.startswith("sensor_")]
drop_cols = [
    "sensor_1", "sensor_5", "sensor_6", "sensor_10",
    "sensor_16", "sensor_18", "sensor_19",
]
feature_cols = [c for c in sensor_cols if c not in drop_cols]
assert len(feature_cols) == 14

max_cycle = train_df.groupby("unit_id")["cycle"].transform("max")
train_df["RUL"] = max_cycle - train_df["cycle"]
train_df["RUL_clipped"] = train_df["RUL"].clip(upper=RUL_CAP)

test_last = (
    test_df.sort_values(["unit_id", "cycle"])
    .groupby("unit_id", as_index=False)
    .tail(1)
    .reset_index(drop=True)
)
test_last["RUL"] = rul_df["RUL"].values
test_last["RUL_clipped"] = test_last["RUL"].clip(upper=RUL_CAP)

log(f"train {train_df.shape} engines={train_df['unit_id'].nunique()}")
log(f"test  {test_df.shape} engines={test_df['unit_id'].nunique()}")
log(f"NaNs train={train_df.isna().sum().sum()} test={test_df.isna().sum().sum()}")
log(f"14 sensors: {feature_cols}")


train (20631, 28) engines=100
test  (13096, 26) engines=100
NaNs train=0 test=0
14 sensors: ['sensor_2', 'sensor_3', 'sensor_4', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_17', 'sensor_20', 'sensor_21']


In [6]:

def make_train_windows(df, window, features, target="RUL_clipped"):
    X, y, units = [], [], []
    for uid, g in df.groupby("unit_id", sort=True):
        vals = g[features].values
        rul = g[target].values
        for end in range(window, len(g) + 1):
            X.append(vals[end - window : end])
            y.append(rul[end - 1])
            units.append(uid)
    return (
        np.array(X, dtype="float32"),
        np.array(y, dtype="float32"),
        np.array(units),
    )


def make_test_windows(df, window, features):
    X = []
    for uid, g in df.groupby("unit_id", sort=True):
        vals = g[features].values
        if len(vals) < window:
            pad = np.repeat(vals[:1], window - len(vals), axis=0)
            vals = np.vstack([pad, vals])
        X.append(vals[-window:])
    return np.array(X, dtype="float32")


# --- Protocol step 1: total windows (unscaled features) ---
X_all, y_all, units_all = make_train_windows(train_df, WINDOW, feature_cols, target="RUL_clipped")
expected = len(train_df) - train_df["unit_id"].nunique() * (WINDOW - 1)
log("=== Window counts and engine split (protocol 1) ===")
log(f"Total training rows: {len(train_df)}")
log(f"Expected windows (rows - 100*29): {expected}")
log(f"Actual windows: {len(X_all)} shape {X_all.shape[1:]}")
assert len(X_all) == 17731, len(X_all)
assert X_all.shape[1:] == (30, 14)
assert not np.isnan(X_all).any()
log("Assert total windows == 17731: OK")
log("Assert window shape (30, 14) and no NaNs: OK")

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_idx, va_idx = next(gss.split(X_all, y_all, groups=units_all))
train_engines = set(units_all[tr_idx])
val_engines = set(units_all[va_idx])
log(f"Train engines: {len(train_engines)} | windows: {len(tr_idx)}")
log(f"Val engines:   {len(val_engines)} | windows: {len(va_idx)}")
assert train_engines.isdisjoint(val_engines)
assert len(tr_idx) + len(va_idx) == 17731
log("Assert train+val windows == 17731 and engine sets disjoint: OK")


=== Window counts and engine split (protocol 1) ===
Total training rows: 20631
Expected windows (rows - 100*29): 17731
Actual windows: 17731 shape (30, 14)
Assert total windows == 17731: OK
Assert window shape (30, 14) and no NaNs: OK
Train engines: 80 | windows: 14241
Val engines:   20 | windows: 3490
Assert train+val windows == 17731 and engine sets disjoint: OK


In [7]:

log("=== Scaling (protocol 2): fit MinMax on training-split engines only ===")
scaler = MinMaxScaler()
train_mask = train_df["unit_id"].isin(train_engines)
scaler.fit(train_df.loc[train_mask, feature_cols])

train_scaled = train_df.copy()
test_scaled = test_df.copy()
train_scaled[feature_cols] = scaler.transform(train_df[feature_cols])
test_scaled[feature_cols] = scaler.transform(test_df[feature_cols])

X_train, y_train, units_train = make_train_windows(
    train_scaled, WINDOW, feature_cols, target="RUL_clipped"
)
assert len(X_train) == 17731
assert np.array_equal(units_train, units_all)

X_tr, y_tr = X_train[tr_idx], y_train[tr_idx]
X_va, y_va = X_train[va_idx], y_train[va_idx]
X_test = make_test_windows(test_scaled, WINDOW, feature_cols)
y_test_raw = test_last["RUL"].values.astype("float32")
y_test_clipped = test_last["RUL_clipped"].values.astype("float32")
assert X_test.shape == (100, WINDOW, 14)
log(f"Regenerated arrays: X_tr {X_tr.shape} X_va {X_va.shape} X_test {X_test.shape}")


=== Scaling (protocol 2): fit MinMax on training-split engines only ===
Regenerated arrays: X_tr (14241, 30, 14) X_va (3490, 30, 14) X_test (100, 30, 14)


In [8]:

def rmse(y, p):
    return float(np.sqrt(np.mean((p - y) ** 2)))


def mae(y, p):
    return float(np.mean(np.abs(p - y)))


def mean_signed_error(y, p):
    return float(np.mean(p - y))


def nasa_score(y, p):
    d = p - y
    return float(np.sum(np.where(d < 0, np.exp(-d / 13) - 1, np.exp(d / 10) - 1)))


def metrics_dict(y, p):
    p = np.asarray(p, dtype=float).ravel()
    y = np.asarray(y, dtype=float).ravel()
    return {
        "rmse": rmse(y, p),
        "mae": mae(y, p),
        "mean_signed_error": mean_signed_error(y, p),
        "nasa": nasa_score(y, p),
    }


def log_metrics(name, y, p):
    m = metrics_dict(y, p)
    log(
        f"{name}: RMSE={m['rmse']:.2f} MAE={m['mae']:.2f} "
        f"mean_signed_error={m['mean_signed_error']:.2f} NASA={m['nasa']:.1f}"
    )
    return m


results_rows = []


In [9]:

log("=== Ridge baseline (alpha=1.0, last-cycle 14 sensors) ===")
ridge = Ridge(alpha=1.0).fit(X_tr[:, -1, :], y_tr)
p_va = ridge.predict(X_va[:, -1, :])
p_test = ridge.predict(X_test[:, -1, :])

for label, y, p in [
    ("Ridge validation (target: clipped)", y_va, p_va),
    ("Ridge test (target: RAW)", y_test_raw, p_test),
    ("Ridge test (target: clipped)", y_test_clipped, p_test),
]:
    m = log_metrics(label, y, p)
    split = "validation" if "validation" in label else "test"
    target = "clipped" if "clipped" in label else "raw"
    results_rows.append({"model": "Ridge", "split": split, "target": target, **m})

log(
    "Note: earlier Ridge in 01_data_exploration used MinMax fit on all 100 train engines "
    "before split; numbers here may differ for that reason."
)


=== Ridge baseline (alpha=1.0, last-cycle 14 sensors) ===
Ridge validation (target: clipped): RMSE=19.97 MAE=15.84 mean_signed_error=0.40 NASA=23761.0
Ridge test (target: RAW): RMSE=21.61 MAE=17.33 mean_signed_error=1.97 NASA=1087.2
Ridge test (target: clipped): RMSE=20.41 MAE=16.32 mean_signed_error=3.04 NASA=1011.7
Note: earlier Ridge in 01_data_exploration used MinMax fit on all 100 train engines before split; numbers here may differ for that reason.


In [ ]:

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

log("=== LSTM baseline ===")


def build_lstm():
    inp = keras.Input(shape=(WINDOW, len(feature_cols)))
    x = layers.LSTM(64, return_sequences=True)(inp)
    x = layers.Dropout(0.2)(x)
    x = layers.LSTM(32)(x)
    x = layers.Dense(32, activation="relu")(x)
    out = layers.Dense(1)(x)
    return keras.Model(inp, out)


def train_lstm(seed: int):
    tf.keras.utils.set_random_seed(seed)
    model = build_lstm()
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss="mse")
    cbs = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=10, restore_best_weights=True
        ),
        keras.callbacks.ReduceLROnPlateau(
            factor=0.5, patience=4, min_lr=1e-5
        ),
    ]
    t0 = time.time()
    hist = model.fit(
        X_tr,
        y_tr,
        validation_data=(X_va, y_va),
        epochs=60,
        batch_size=128,
        callbacks=cbs,
        verbose=0,
    )
    elapsed = time.time() - t0
    return model, hist, elapsed


model, history, elapsed = train_lstm(42)
from pathlib import Path

Path("models").mkdir(exist_ok=True)
model.save("models/lstm_rul_fd001.keras")
print("LSTM model saved successfully.")
log(f"LSTM seed 42 training time: {elapsed:.1f}s epochs run: {len(history.history['loss'])}")

p_va_lstm = model.predict(X_va, verbose=0).ravel()
m = log_metrics("LSTM validation (target: clipped)", y_va, p_va_lstm)
results_rows.append(
    {"model": "LSTM", "split": "validation", "target": "clipped", **m}
)

if elapsed < 300:
    log("Run time < 5 min per seed; also training seeds 43 and 44 for test metrics.")
    seeds = [42, 43, 44]
    raw_rows, clip_rows, test_preds = [], [], []
    for seed in seeds:
        if seed == 42:
            mdl = model
        else:
            mdl, _, t_s = train_lstm(seed)
            log(f"LSTM seed {seed} training time: {t_s:.1f}s")
        p = mdl.predict(X_test, verbose=0).ravel()
        test_preds.append(p)
        raw_rows.append(metrics_dict(y_test_raw, p))
        clip_rows.append(metrics_dict(y_test_clipped, p))
    p_test_lstm = np.mean(test_preds, axis=0)

    def log_mean_sd(name, rows, key):
        vals = [r[key] for r in rows]
        log(f"{name} {key}: {np.mean(vals):.2f} +/- {np.std(vals, ddof=1):.2f} (seeds {seeds})")

    for key in ["rmse", "mae", "mean_signed_error", "nasa"]:
        log_mean_sd("LSTM test (target: RAW)", raw_rows, key)
        log_mean_sd("LSTM test (target: clipped)", clip_rows, key)

    m_raw = metrics_dict(y_test_raw, p_test_lstm)
    m_clip = metrics_dict(y_test_clipped, p_test_lstm)
    log_metrics("LSTM test (target: RAW) — mean prediction across seeds", y_test_raw, p_test_lstm)
    log_metrics(
        "LSTM test (target: clipped) — mean prediction across seeds",
        y_test_clipped,
        p_test_lstm,
    )
    results_rows.append({"model": "LSTM", "split": "test", "target": "raw", **m_raw})
    results_rows.append({"model": "LSTM", "split": "test", "target": "clipped", **m_clip})
else:
    log("Single seed 42 only (first run exceeded 5 min threshold for multi-seed).")
    p_test_lstm = model.predict(X_test, verbose=0).ravel()
    for label, y in [
        ("LSTM test (target: RAW)", y_test_raw),
        ("LSTM test (target: clipped)", y_test_clipped),
    ]:
        m = log_metrics(label, y, p_test_lstm)
        target = "raw" if "RAW" in label else "clipped"
        results_rows.append({"model": "LSTM", "split": "test", "target": target, **m})

lstm_history = history
ridge_test_pred = p_test
lstm_test_pred = p_test_lstm


=== LSTM baseline ===


In [ ]:

log("=== Saving plots and reports ===")

# 1) True vs predicted (test, RAW primary)
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test_raw, lstm_test_pred, alpha=0.7, label="LSTM", s=40)
ax.scatter(y_test_raw, ridge_test_pred, alpha=0.5, label="Ridge", s=40)
lim = max(y_test_raw.max(), lstm_test_pred.max(), ridge_test_pred.max()) + 5
ax.plot([0, lim], [0, lim], "k--", lw=1, label="y=x")
ax.set_xlabel("True RUL")
ax.set_ylabel("Predicted RUL")
ax.set_title("Test engines — target: RAW RUL")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "true_vs_pred_test_raw.png", dpi=150)
plt.close(fig)

# 2) Loss curves (LSTM, clipped training target)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(lstm_history.history["loss"], label="train")
ax.plot(lstm_history.history["val_loss"], label="validation")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE loss")
ax.set_title("LSTM training — target: clipped (min(RUL,125))")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "lstm_loss_curves.png", dpi=150)
plt.close(fig)

# 3) Residual histogram (LSTM test RAW)
resid = lstm_test_pred - y_test_raw
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(resid, bins=20, edgecolor="black")
ax.axvline(0, color="k", ls="--")
ax.set_xlabel("pred - true (positive = late)")
ax.set_ylabel("Count")
ax.set_title("LSTM residuals — test target: RAW RUL")
fig.tight_layout()
fig.savefig(FIG_DIR / "residual_hist_lstm_test_raw.png", dpi=150)
plt.close(fig)

results_df = pd.DataFrame(results_rows)
csv_path = REPORTS / "results.csv"
results_df.to_csv(csv_path, index=False)
log(f"Wrote {csv_path}")

md_lines = [
    "# AI-based Predictive Maintenance for Prediction of Simulated Turbofan Engine",
    "",
    "NASA C-MAPSS FD001, LSTM-based RUL prediction",
    "",
    "Mean signed error = mean(pred − true): **positive** = late (over-predicting RUL); **negative** = early.",
    "",
    "## Metrics (from run_log.txt)",
    "",
    "| Model | Split | Target | RMSE | MAE | Mean signed error | NASA |",
    "| --- | --- | --- | --- | --- | --- | --- |",
]
for _, row in results_df.iterrows():
    md_lines.append(
        f"| {row['model']} | {row['split']} | {row['target']} | "
        f"{row['rmse']:.2f} | {row['mae']:.2f} | {row['mean_signed_error']:.2f} | {row['nasa']:.1f} |"
    )
md_lines.extend(
    [
        "",
        "## Ridge comparison to notebook 01",
        "",
        "Notebook `01_data_exploration.ipynb` fit MinMaxScaler on all 100 training engines before the engine split; "
        "this notebook fits the scaler on training-split engines only (protocol). Validation/test metrics may differ.",
        "",
        "Reference (01, all-engine scaler): test RMSE 21.61 / NASA 1087.2 vs **raw** RUL; "
        "20.41 / 1011.7 vs **clipped**; validation RMSE 19.97 vs clipped.",
        "",
        "## Figures",
        "",
        "- `figures/true_vs_pred_test_raw.png`",
        "- `figures/lstm_loss_curves.png`",
        "- `figures/residual_hist_lstm_test_raw.png`",
    ]
)
md_path = REPORTS / "results.md"
md_path.write_text("\n".join(md_lines) + "\n", encoding="utf-8")
log(f"Wrote {md_path}")
log("Done.")


=== Saving plots and reports ===


Wrote C:\Users\samya\Desktop\mini_project\reports\results.csv
Wrote C:\Users\samya\Desktop\mini_project\reports\results.md
Done.
